## Workbook 2: Pandas and SQL

All resources, unless otherwise specified, are created by Nikhil Chinchalkar

`You've already seen some of the stuff we can do with Pandas, and we're going to expand on that with some more advanced filtering and general data cleaning skills. We'll also learn the basics of SQL (which is another language), and compare using SQL and Pandas to perform basic filtering tasks.

### A: Filtering In Pandas Vs. SQL

We saw earlier that we can do some basic filtering based on columns using some basic Pandas functions. We can also do the same stuff using SQL (and in my opinion, it's much easier). Let's start by importing `duckdb`, which is a library that allows us to run SQL (which is a different language) queries in Python.

In [3]:
import numpy as np
import pandas as pd
import duckdb

If the above gave you an error, try commenting out the line below and re-running it once the stuff below has finished.

In [ ]:
# %pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 26.3 MB/s  0:00:00m0:00:010:01
Note: you may need to restart the kernel to use updated packages.


We'll start by importing data from [Netflix TV Shows and Movies](https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies?select=titles.csv), which is available on Kaggle. This dataset shows information for all shows and movies that were available to Netflix users in July 2022.

In [4]:
# we use the pandas (abbreviated to pd) function `read_csv` to input the data, which is stored under the data folder 
titles = pd.read_csv('data/titles.csv')

# print the first 3 rows
titles.head(3)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300


For now, let's just do some random filtering and grouping to show the difference between Pandas and SQL. Let's say we only wanted to look at TV Shows with more than 1 season and only cared about the columns `id` and `title`.

In [5]:
#pandas first:

# only look at titles that are of `type` 'SHOW' with more than 1 `seasons`, then only look at two columns
filtered_titles_PD = titles[(titles['type'] == 'SHOW') & (titles['seasons'] > 1)][['id','title']]

print(filtered_titles_PD.head())

         id                         title
5   ts22164  Monty Python's Flying Circus
35  ts20681                      Seinfeld
44  ts22082                  Knight Rider
45  ts21715              Thomas & Friends
46  ts20678             Saved by the Bell


That above Pandas syntax is a little weird and unintuitive. It doesn't exactly fly off the tongue either. Let's see the same exact task in SQL. First, I'll explain what the SQL syntax looks like. Most queries will start with a `SELECT` command, which just tells Python to take some columns `FROM` a dataframe, which is our next command. Then, there's a bunch of syntax that can follow -- I'm using a `WHERE` call to specify that I want to take data *where* a condition is satisfied. That condition? Where `type = 'SHOW' AND seasons > 1`. Note that the surrounding structure is important - you need the initial `duckdb.sql`, the triple quotes, and the `df()` following to convert the task back into a dataframe.

Note that we tend to capitalize functions in SQL and leave other stuff lowercase - there's no difference in not doing that, it just makes it a bit easier to read.

In [6]:
#SQL next:
# select the id, title columns from the titles dataframe, and rows only where the condition is true
filtered_titles_SQL = duckdb.sql("""SELECT id, title 
                                 FROM titles
                                 WHERE type = 'SHOW' AND seasons > 1""").df()

print(filtered_titles_SQL.head())

        id                         title
0  ts22164  Monty Python's Flying Circus
1  ts20681                      Seinfeld
2  ts22082                  Knight Rider
3  ts21715              Thomas & Friends
4  ts20678             Saved by the Bell


Both the above tasks accomplished the same thing - but one is significantly easier than the other (SQL). I'm going to continue to use SQL for most of my data cleaning tasks, including grouping, which is next:

Grouping is a bit tricky - you take an item in a column and put all the rows that match it together. Then you can select attributes about each group, where each of those "attributes" must correspond to a single value. Let's say we wanted to group by `SHOW`s and `MOVIE`s to see how many of each our original dataset has, along with their rating:

In [7]:
# select the count and avg imdb score of each media type in the dataframe
titles_count = duckdb.sql("""SELECT type, COUNT(*) AS Count, AVG(imdb_score) AS AverageScore 
                          FROM titles
                          GROUP BY type""").df()

print(titles_count)

    type  Count  AverageScore
0  MOVIE   3744      6.246748
1   SHOW   2106      6.977927


Though the result is pretty simple, there's a lot going on in the above call. It's slightly easier to parse through the SQL query in reverse. So, from the `titles` dataframe, we create groups based on unique values of `type`. Then for each of those groups, we select some information to print out as columns. That information is the `type` itself, as well as the `COUNT(*)` (where `*` selects all columns) of the number of rows that are in each group, and the average IMDB score for each title in each group. Note that these columns have only one value for each group - though each `type` corresponds to thousands of rows, they can only be represented as a single row in the final output. Hence, we need to *aggregate* information from each group into a single value, and functions like `COUNT` and `AVG` do just that.

The `AS` keyword is basically us renaming the column to something easier to interpret - you can remove the `AS` command to see how our output changes. 

Take a second to really understand what's going on in the above call - a lot of this stuff are things that we need to do for every dataset when doing exploratory data analysis.

It's now your turn: try and do the following from the `credits` csv that I downloaded from Kaggle. Completing this step will involve some stuff from prior notebooks too.

1. Read in the CSV, saving it in a variable titled `credits`
2. Using SQL do the following (from the `credits` dataframe):
   1. Group by both `person_id` and `name`
      1. We group by `person_id` to separate out actors with the same name, but also group by `name` since one person might have multiple names
   2. Count the number of roles that each actor `person_id` is attributed to (similar to what we did above), renaming the column to `CountRoles`
   3. Use the [`ORDER BY`](https://www.w3schools.com/sql/sql_orderby.asp) command to sort the `CountRoles` in descending order
   4. Make sure to select the `name` and `CountRoles` columns only
   5. Save this dataframe to a variable titled `topActors`
3. Using Pandas, print the first 5 rows of `topActors`

In [16]:
#A1 your code here
#part 1 is already done for you, along with some skeleton code
credits = pd.read_csv('data/credits.csv')

topActors = duckdb.sql("""SELECT name, COUNT(*) as CountRoles
                       FROM credits
                       GROUP BY person_id, name
                       ORDER BY CountRoles DESC
                       """).df()

topActors.head(5)

,name,CountRoles
0,Boman Irani,25
1,Kareena Kapoor Khan,25
2,Shah Rukh Khan,23
3,Takahiro Sakurai,21
4,Priyanka Chopra Jonas,20


Expected Output:
```
                  name  CountRoles
0  Kareena Kapoor Khan          25
1          Boman Irani          25
2       Shah Rukh Khan          23
3     Takahiro Sakurai          21
4          Raúl Campos          20
```

The above section details pretty much all of the basic SQL commands, there's obviously more, but for our sake, that's a pretty solid foundation. There's one more important SQL command, though, that we need to address. Before we do that, we'll talk about `NaN`s.

### B: Dealing With NANs

`NaN`s, or 'Not a Number' is a type of value in Python that exists when we have a 'blank' in a dataset. Essentially, we see an `NaN` in a dataset when there is no data to fill in that value. We can see an `NaN` in practice by looking at the bottom of the `credits` dataframe, in the `character` column. Here, the character that Julian Gaviria plays is unknown/null.

In [17]:
credits.tail(5)

,person_id,id,name,character,role
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR
77800,1950416,tm1059008,Julian Gaviria,NaN,DIRECTOR


`NaN`s can be a problem for certain applications and calculations. You can't take the mean of a column with `NaN`s in it, and graphing `NaN`s can look a bit weird. 

Dealing with `NaN`s is a large part of the data cleaning process, and is something that depends on your specific application. Sometimes you might want to filter out the `NaN`s in one column, and leave in `NaN`s in another, and sometimes you might want to replace `NaN`s with actual values like 0s. For now, I'll just show you how to remove all `NaN`s in all columns, which is useful for regression applications, since most models require no `NaN`s as inputs.

To do so, we just use Pandas and a simple command.

In [19]:
#remove all rows that have an NaN in any column
credits_cleaned = credits.dropna()
credits_cleaned.tail(5)

,person_id,id,name,character,role
77795,368473,tm1059008,Aída Morales,Maritza,ACTOR
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


Now we can see that there's no longer that `NaN` at the bottom of the list. Note that removing certain rows can cause the row indices to get messed up, since there's no longer going to be rows at certain indices. To fix that and ensure our rows are continuous, we can just call `reset_index`, setting `drop=True` to make sure we don't add another column.

In [20]:
# reset the index without adding a new column for the previous index
credits_cleaned = credits_cleaned.reset_index(drop=True)
credits_cleaned.tail(5)

,person_id,id,name,character,role
68024,368473,tm1059008,Aída Morales,Maritza,ACTOR
68025,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
68026,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
68027,373198,tm1059008,Inés Prieto,Fanny,ACTOR
68028,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


We can now see that our row indices have reset, and while it's a bit difficult to show that they're continuous, just trust me that they are.

### C: Left Joins

The last SQL command we'll learn is arguably the most important one. It deals with when we want to merge two datasets together (horizontally). You might have noticed that we have two separate but similar datasets, that can be merged *on* the `id` column. That is, each actor role corresponds to a movie `id`, since each actor plays a role in a movie. We can merge the two datasets `credits` and `titles` to share that information. 

There's a bunch of ways to do that, but the easiest is to perform a `LEFT JOIN`, which means we have two datasets and a column to combine on (the column shares the same data). Then, for each row in the right dataset, we join that row to every instance of the value in the left dataset. It's easier to see what I mean in practice.

There's a way of doing this using Pandas, but like we saw above, that method is a bit unintuitive, so I'll use SQL:

In [44]:
# left join titles with credits, so each movie's id gets matched with all the actors had a role with that movie id
merged_dataset = duckdb.sql("""SELECT titles.id AS titles_id, credits.id AS credits_id, titles.title, 
                            titles.imdb_score, credits.name, credits.character 
                            FROM titles
                            LEFT JOIN credits ON titles.id = credits.id""").df()

merged_dataset

,titles_id,credits_id,title,imdb_score,name,character
0,tm84618,tm84618,Taxi Driver,8.2,Robert De Niro,Travis Bickle
1,tm84618,tm84618,Taxi Driver,8.2,Jodie Foster,Iris Steensma
2,tm84618,tm84618,Taxi Driver,8.2,Albert Brooks,Tom
3,tm84618,tm84618,Taxi Driver,8.2,Harvey Keitel,Matthew 'Sport' Higgins
4,tm84618,tm84618,Taxi Driver,8.2,Cybill Shepherd,Betsy
...,...,...,...,...,...,...
78157,ts225708,None,Six Windows in the Desert,4.3,None,None
78158,tm1080554,None,Rudra: Secret of the Black Moon,NaN,None,None
78159,ts319333,None,Tiger King: The Doc Antle Story,6.2,None,None
78160,tm879685,None,The Perfect Family,5.3,None,None


In the above cell, there's a lot going on. First, we're using some new notation to help Python identify which columns we actually want to take and which dataframes they actually come from (i.e. using `titles.` and `credits.`). Then, we see some familiar syntax in our `FROM` statement, before we're met with another new function in `LEFT JOIN`. Here, we're just saying that we want to left join starting from `titles` (the first dataframe is the left one) `ON` the common column which corresponds to `id` in `titles` and also just happens to be `id` in `credits`. Note that they don't need to be the same name, it just so happens to be like that in this case. 

Our output here is as expected: we see that each of the rows in the left dataframe (a movie) is being matched to *all* the corresponding rows in the right dataframe (actors). In other words, each actor role is being matched with the movie that the role comes from. Since there are multiple roles in each movie, it makes sense that we see Taxi Driver a bunch of times, for instance. Also note the bottom of the dataframe. `titles` has some movies that did not have any listed actors, so nothing matches from the `credits` dataframe, and the columns from `credits` are all empty (`None`). 

An artifact of this left join is that any actors that played roles in movies that did not exist in the `titles` dataframe would no longer be in the `merged_dataset` dataframe, since they had no rows to match with.

While this might seem a bit complicated right now, it should become second nature with some practice. Also note that there are a ton of other merging methods (right joins, inner joins, full joins, etc.) but you'll rarely see these other types of joins. Besides, with some manipulation you could get the equivalent of a right join with a left join, and vice versa. Essentially, yeah there's a bunch of joins, but they all can reach the same goal, which is just to have one merged dataset.

We just went over a lot of stuff, but that's pretty much all the SQL commands that I've had to use so far for projects, with some minor tweaks to syntax here and there. It's now time to do your own practice:

In the following code block, do the following:

1. Make a new dataframe, called `directors`, from the `credits` dataframe, only looking at directors (filter on the `role` column)
2. Perform a left join on the `directors` dataframe (using `titles` as the right dataframe) to match up each director to the film they directed (this should look similar to the above)
3. Use the `GROUP BY` command to group by each director's `name`
4. Use the `COUNT()` function to list the number of movies each director directed (rename this to `count`)
5. Use the `AVG()` function to find the average `imdb_score` of each of the movies (rename this to `score`)
6. Print the first 5 rows of `directors`

In [38]:
#C1 your code here
# skeleton code provided

directors = duckdb.sql("""SELECT name, COUNT(*) as count, AVG(imdb_score) as score 
                       FROM credits
                       LEFT JOIN titles ON credits.id = titles.id
                       WHERE role = 'DIRECTOR'
                       GROUP BY name""").df()

directors.head(5)

,name,count,score
0,Prakash Jha,4,7.050
1,Soi Cheang,1,6.600
2,Ayan Mukerji,1,7.600
3,Abhishek Chaubey,4,6.875
4,Renzil D'Silva,2,5.750


Expected Output (columns should match, but data may differ):

```
                 name  count  score
0        David Zeiger      1   7.60
1  Christopher Martin      1   5.90
2       Milan Luthria      6   5.80
3         Zoya Akhtar      6   7.04
4          Soham Shah      1   4.70
```

6. Make a new dataframe from `directors` titled `top_directors` where you filter the `directors` who directed at least 5 movies
7. Sort `top_directors` by `score`, in descending order
8. Print the first 5 rows of `top_directors`

In [40]:
#C2 your code here
# skeleton code provided

top_directors = duckdb.sql("""SELECT name, count, score 
                           FROM directors
                           WHERE count >= 5
                           ORDER BY score DESC""").df()

top_directors.head(5)

,name,count,score
0,Martin Scorsese,6,8.160000
1,Hong Jong-chan,5,7.900000
2,Mahdi Fleifel,5,7.800000
3,Mani Ratnam,6,7.733333
4,Mike Clattenburg,5,7.550000


Expected Output:

```
              name  count     score
0  Martin Scorsese      6  8.160000
1      Mani Ratnam      6  7.733333
2   Song Hyun-wook      6  7.375000
3   Anurag Kashyap      8  7.271429
4  Shannon Hartman      7  7.257143
```

This last assignment is pretty difficult and requires a solid foundation in SQL. If you completed it, you should feel confident about your ability to solve future problems you encounter in SQL. If you completed it and still want a challenge, try to do it in a single SQL query (hint: you should look into the [`HAVING`](https://www.w3schools.com/sql/sql_having.asp) function).

Now, try and make a estimation for a frequency distribution of scores across our `titles` dataset. To do this, we'll need to do the following:

1. Select the `CEIL()` of the `imdb_score`s from the `titles` dataframe (rename this to `score`)
2. Select the `COUNT()` of the number of movies that fall under each score (rename this to `freq`)
3. `GROUP BY` each score
4. Sort the dataframe in descending order by the `score` attribute
5. Remove `NaN`s in the score column using the `dropna()` function (via `Pandas`)
6. Display the full dataframe

In [42]:
#C3 your code here
# skeleton code provided

c3 = duckdb.sql("""SELECT CEIL(imdb_score) as score, COUNT(*) as freq
                  FROM titles
                  GROUP BY score 
                  ORDER BY score DESC""").df()

clean_c3 = c3.dropna()
clean_c3

,score,freq
0,10.0,11
1,9.0,390
2,8.0,1490
3,7.0,1835
4,6.0,1094
5,5.0,376
6,4.0,125
7,3.0,39
8,2.0,8


Expected Output:

```
   score  freq
0   10.0    11
1    9.0   390
2    8.0  1490
3    7.0  1835
4    6.0  1094
5    5.0   376
6    4.0   125
7    3.0    39
8    2.0     8
```

After the next lesson, you'll easily be able to plot this data in a nice histogram plot.